In [ ]:
import numpy as np
print("All modules imported successfully!")

In [ ]:
def sigmoid(X):
    X = np.clip(X, -500, 500)
    return 1 / (1 + np.exp(-X))
def softmax(X):
    e_x = np.exp(X - np.max(X))
    return e_x / e_x.sum(axis=0)

In [ ]:
class LSTM:
    def __init__(self, input_size, hidden_size, output_size,learning_rate=0.01):
        self.is_dim = input_size
        self.hs = hidden_size
        self.lr = learning_rate

        concat_size = self.is_dim + self.hs 
        scale = 1.0 / np.sqrt(self.hs)

        self.weights = {
            'Wf': np.random.randn(self.hs, concat_size) * scale,
            'bf': np.ones((self.hs, 1)),

            'Wi': np.random.randn(self.hs, concat_size) * scale,
            'bi': np.zeros((self.hs, 1)),

            'Wc': np.random.randn(self.hs, concat_size) * scale,
            'bc': np.zeros((self.hs, 1)),


            'Wo': np.random.randn(self.hs, concat_size) * scale,
            'bo': np.zeros((self.hs, 1)),

            'Wy': np.random.randn(output_size, self.hs) * scale,
            'by': np.zeros((output_size, 1)),
        }

    def forward(self, X):
        """
        X: list or array of input vectors over time. Shape: (seq_len, input_size, 1)
        """
        seq_len = len(X)

        H = {}
        C = {}
        C_tilde = {}
        f, i, O = {}, {}, {}
        y_preds = {}

        H[-1] = np.zeros((self.hs, 1))
        C[-1] = np.zeros((self.hs, 1))

        for t in range(seq_len):
            concat_input = np.vstack((H[t-1], X[t]))
            f[t] = sigmoid(np.dot(self.weights['Wf'], concat_input) + self.weights['bf'])
            i[t] = sigmoid(np.dot(self.weights['Wi'], concat_input) + self.weights['bi'])
            C_tilde[t] = np.tanh(np.dot(self.weights['Wc'], concat_input) + self.weights['bc'])
            O[t] = sigmoid(np.dot(self.weights['Wo'], concat_input) + self.weights['bo'])

            C[t] = (f[t] * C[t-1]) + (i[t] * C_tilde[t])
            H[t] = O[t] * np.tanh(C[t])
            y_preds[t] = np.dot(self.weights['Wy'], H[t]) + self.weights['by']

        caches = (X, H, C, C_tilde, f, i, O)
        return y_preds, caches

    def backward(self, y_preds, caches, y_true):
        X, H, C, C_tilde, f, i, O = caches
        seq_len = len(X)

        grads = {k: np.zeros_like(v) for k, v in self.weights.items()}
        dh_next = np.zeros((self.hs, 1))
        dc_next = np.zeros((self.hs, 1))

        for t in reversed(range(seq_len)):
            dy = y_preds[t] - y_true[t]

            grads['Wy'] += np.dot(dy, H[t].T)
            grads['by'] += dy

            dht_local = np.dot(self.weights['Wy'].T, dy) 
            dht = dht_local + dh_next

            dct_local = dht * O[t] * (1 - np.tanh(C[t]) ** 2)
            dct = dct_local + dc_next

            concat_input = np.vstack((H[t-1], X[t]))

            dZo = (dht * np.tanh(C[t])) * (O[t] * (1 - O[t]))
            dZf = (dht * C[t-1]) * (f[t] * (1 - f[t]))
            dZi = (dht * C_tilde[t]) * (i[t] * (1 - i[t]))
            dZc = (dct * i[t]) * (1 - C_tilde[t] ** 2)

            grads['Wf'] += np.dot(dZf, concat_input.T); grads['bf'] += dZf
            grads['Wi'] += np.dot(dZi, concat_input.T); grads['bi'] += dZi
            grads['Wc'] += np.dot(dZc, concat_input.T); grads['bc'] += dZc
            grads['Wo'] += np.dot(dZo, concat_input.T); grads['bo'] += dZo

            dZ_concat = np.dot(self.weights['Wf'].T, dZf) + \
                        np.dot(self.weights['Wi'].T, dZi) + \
                        np.dot(self.weights['Wc'].T, dZc) + \
                        np.dot(self.weights['Wo'].T, dZo)
            
            dh_next = dZ_concat[:self.hs, :]
            dc_next = dct * f[t]

        for key in grads:
            grads[key] = np.clip(grads[key], -5, 5)

        return grads

    def update_params(self, grads):
        for key in self.weights.keys():
            self.weights[key] -= self.lr * grads[key]

    def train(self, dataset, epochs=100):
        for epoch in range(epochs):
            total_loss = 0
            for x_seq, y_seq in dataset:
                y_preds, caches = self.forward(x_seq)
                seq_loss = 0
                for t in range(len(x_seq)):
                    seq_loss += -np.sum(y_seq[t] * np.log(y_preds[t] + 1e-8))
                total_loss += seq_loss
                grads = self.backward(y_preds, caches, y_seq)
                self.update_params(grads)
            if (epoch + 1) % 10 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataset):.4f}')

     

    